# RECAP vs BRICS on one peptide

Both are **slicers** — rules for which bonds get cut when a molecule is turned into a
SAFE string. They run *before* tokenization, so they change the text itself, and each
one needs its own tokenizer trained on its own corpus.

This notebook walks one real antimicrobial peptide through both paths and compares
what comes out. Runs from anywhere inside the repo.

In [ ]:
import os, sys, textwrap, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Jupyter starts the kernel in the notebook's own directory, not the repo root,
# so locate the root and move there -- every path below is root-relative.
ROOT = Path.cwd()
while not (ROOT / "scripts" / "tokenizer_report.py").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError(f"repo root not found from {Path.cwd()}")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
print("repo root:", ROOT)

import pandas as pd
from rdkit import Chem, RDLogger
from safe import SAFEConverter, SAFETokenizer
import safe as sf

RDLogger.DisableLog("rdApp.*")

# the same span math the sweep report uses, so the numbers here agree with it
from tokenizer_report import ring_events, char_to_token

AMP_CSV  = "molecular_dataset/dataset/data/dbaasp/amp.csv"
TOK = {
    "brics": "molecular_dataset/dataset/data/safe/tokenizer.json",  # trained on the brics corpus
    "recap": "tokenizers/tok_recap_safe.json",                      # trained on the recap corpus
}
print("ready")

repo root: /home/raymondlab/Documents/MaskedDiffusionAMP
ready


## The peptide

Dermaseptin S4 (3-15) — a 13-residue AMP. Nothing special about it except that it is
real, it is in the training corpus, and both slicers handle it.

In [ ]:
df = pd.read_csv(AMP_CSV)
row = df[df.amp_id == 31].iloc[0]

print(f"name     {row['name']}")
print(f"sequence {row.sequence}  ({len(row.sequence)} residues)")
print(f"atoms    {Chem.MolFromSmiles(row.smiles).GetNumAtoms()}")
print()
print("SMILES")
print(textwrap.fill(row.smiles, 92, subsequent_indent="  "))

name     Dermaseptin S4 (3-15)AMD[M4K]
sequence WKTLLKKVLKAAA  (13 residues)
atoms    104

SMILES
CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](NC(=O)[C@H](CCCCN)NC(=O)[C@@H](N)Cc1c[nH]c2ccccc
  12)[C@@H](C)O)C(=O)N[C@@H](CCCCN)C(=O)N[C@@H](CCCCN)C(=O)N[C@H](C(=O)N[C@@H](CC(C)C)C(=O)N
  [C@@H](CCCCN)C(=O)N[C@@H](C)C(=O)N[C@@H](C)C(=O)N[C@@H](C)C(N)=O)C(C)C


## Step 1 — the slicer cuts bonds

The molecule is a graph. SAFE turns it into text by cutting some bonds, listing the
fragments separated by `.`, and leaving numbered labels behind to record which cut
ends used to be joined.

`brics` uses 16 retrosynthetic rules. `recap` uses 11 reaction patterns, the first two
of which match **amide bonds** — the peptide backbone bond.

In [ ]:
safe_str = {}
for name in ("brics", "recap"):
    s = SAFEConverter(slicer=name).encoder(row.smiles, allow_empty=True)
    safe_str[name] = s
    frags = s.split(".")
    print(f"=== {name} ===")
    print(f"fragments {len(frags):<4} chars {len(s)}")
    print(textwrap.fill(s, 92, subsequent_indent="  "))
    print()

print("identical string?", safe_str["brics"] == safe_str["recap"])

=== brics ===
fragments 26   chars 371
C%21(=O)[C@@H]%10CCCCN.c1%20c[nH]c2ccccc12.[C@H]%11(CCCCN)C%24=O.[C@H]%12(CCCCN)C%25=O.[C@H]
  %16(CCCCN)C4=O.CC(C)C[C@H]7C%23=O.C3(=O)[C@@H]8CC(C)C.[C@H]%15(CC(C)C)C%27=O.C%14(=O)[C@@H
  ]9[C@@H](C)O.[C@@H]%13(C%26=O)C(C)C.C%22(=O)[C@@H](N)C%20.[C@H]%17(C)C5=O.[C@H]%18(C)C6=O.
  [C@H]%19(C)C(N)=O.N37.N%148.N%219.N%22%10.N%23%11.N%24%12.N%25%13.N%26%15.N%27%16.N4%17.N5
  %18.N6%19

=== recap ===
fragments 26   chars 386
C%22=8[C@@H](N)Cc1c[nH]c2ccccc12.C5=%16[C@H](CCCCN)N%22.N%11[C@@H](CCCCN)C%25=%19.N%25[C@@H]
  (CCCCN)C%26=9.N6[C@@H](CCCCN)C=%217.CC(C)C[C@H](N%18)C=%23%11.C=%27%18[C@H](CC(C)C)N%10.N3
  [C@@H](CC(C)C)C=%146.C%10=%20[C@@H](N5)[C@@H](C)O.N%26[C@H](C3=%17)C(C)C.N7[C@@H](C)C=4%13
  .N%13[C@@H](C)C=%24%12.N%12[C@@H](C)C=%15N.O=%27.O=%20.O=%16.O=8.O=%23.O=%19.O=9.O=%17.O=%
  14.O=%21.O=4.O=%24.O=%15

identical string? False


Different text for the same molecule. Both still decode back to the original — the
slicer changes the *representation*, not the chemistry:

In [ ]:
target = Chem.CanonSmiles(row.smiles)
for name, s in safe_str.items():
    back = Chem.CanonSmiles(sf.decode(s, canonical=True, ignore_errors=False))
    print(f"{name:>6}  round-trips to the same molecule: {back == target}")

 brics  round-trips to the same molecule: True
 recap  round-trips to the same molecule: True


## Step 2 — the tokenizer chops the string

Each slicer gets its own tokenizer, trained on its own corpus. Both use
`splitter="safe"`, which pre-splits to single atoms — so these are the atom-level
tokenizers, vocab ~159 either way.

In [ ]:
tokens = {}
for name in ("brics", "recap"):
    tok = SAFETokenizer.load(TOK[name])
    hf  = tok.get_pretrained()
    ids = tok.encode(safe_str[name])
    specials = {t for t in (hf.bos_token, hf.eos_token, hf.cls_token,
                            hf.sep_token, hf.pad_token) if t}
    toks = [t for t in hf.convert_ids_to_tokens(ids) if t not in specials]
    tokens[name] = toks
    print(f"=== {name} ===  vocab {len(tok):<5} tokens {len(toks)}")
    print(textwrap.fill(" ".join(toks[:60]), 92, subsequent_indent="  "))
    print("  ...")
    print()

=== brics ===  vocab 159   tokens 234
C %21 ( = O ) [C@@H] %10 C C C C N . c 1 %20 c [nH] c 2 c c c c c 1 2 . [C@H] %11 ( C C C C
  N ) C %24 = O . [C@H] %12 ( C C C C N ) C %25 = O . [C@H] %16 (
  ...

=== recap ===  vocab 159   tokens 245
C %22 = 8 [C@@H] ( N ) C c 1 c [nH] c 2 c c c c c 1 2 . C 5 = %16 [C@H] ( C C C C N ) N %22
  . N %11 [C@@H] ( C C C C N ) C %25 = %19 . N %25 [C@@H] ( C C C
  ...



## Step 3 — the part that matters: ring-closure pairs

Those numeric labels are **matching brackets**. Every label appears exactly twice, and
the two halves are the same bond. A model that opens `%14` and never closes it has
produced an invalid molecule, the same way an unbalanced paren is an invalid program.

A masked diffusion model fills positions in parallel with no stack, so each pair is a
constraint it must satisfy across however many token positions separate the halves.
**That distance is the span, and it is what we are trying to shrink.**

In [ ]:
import numpy as np

def pair_spans(s, toks):
    """Token distance between the two halves of every ring-closure pair."""
    assert "".join(toks) == s, "tokenization is not character-exact"
    c2t, open_at, spans = char_to_token(toks), {}, []
    for ci, lab in ring_events(s):
        ti = c2t[ci]
        if lab in open_at:
            spans.append(ti - open_at.pop(lab))
        else:
            open_at[lab] = ti
    return spans, open_at

summary = {}
for name in ("brics", "recap"):
    spans, unclosed = pair_spans(safe_str[name], tokens[name])
    summary[name] = dict(fragments=safe_str[name].count(".") + 1,
                         chars=len(safe_str[name]),
                         tokens=len(tokens[name]),
                         pairs=len(spans),
                         median_span=float(np.median(spans)),
                         max_span=int(np.max(spans)),
                         total_span=int(np.sum(spans)))
    print(f"=== {name} ===")
    print(f"  ring pairs   {len(spans)}   (unclosed: {len(unclosed)})")
    print(f"  median span  {np.median(spans):.0f} tokens")
    print(f"  max span     {np.max(spans)} tokens")
    print(f"  total span   {np.sum(spans)} tokens of constraint to satisfy")
    print()

=== brics ===
  ring pairs   27   (unclosed: 0)
  median span  103 tokens
  max span     195 tokens
  total span   2894 tokens of constraint to satisfy

=== recap ===
  ring pairs   27   (unclosed: 0)
  median span  66 tokens
  max span     205 tokens
  total span   2005 tokens of constraint to satisfy



## Summary

In [ ]:
out = pd.DataFrame(summary).T
out.index.name = "slicer"
print(out.to_string())

d = 100 * (1 - summary["recap"]["total_span"] / summary["brics"]["total_span"])
print(f"\nOn this peptide recap cuts total cross-token constraint by {d:.0f}% "
      f"({summary['brics']['total_span']} -> {summary['recap']['total_span']} tokens),")
print("and median span by "
      f"{100*(1-summary['recap']['median_span']/summary['brics']['median_span']):.0f}%. "
      "Note max span is slightly WORSE (195 -> 205): recap shortens the typical")
print("pair, not the single worst one.")

        fragments  chars  tokens  pairs  median_span  max_span  total_span
slicer                                                                    
brics        26.0  371.0   234.0   27.0        103.0     195.0      2894.0
recap        26.0  386.0   245.0   27.0         66.0     205.0      2005.0

On this peptide recap cuts total cross-token constraint by 31% (2894 -> 2005 tokens),
and median span by 36%. Note max span is slightly WORSE (195 -> 205): recap shortens the typical
pair, not the single worst one.


### Reading this

One peptide is an illustration, not evidence. The corpus-wide numbers are what the
comparison rests on — from `scripts/tokenizer_report.py` over the full ~20.6k molecules:

| arm | median length | median span | pairs inside one token | SAFE→SMILES |
|---|---|---|---|---|
| `brics / safe` | 313 | 141 | 0.0% | 100% |
| `recap / safe` | 323 | 115 | 0.0% | 100% |
| `brics / none` | 53 | 29 | 3.2% | 100% |
| `recap / none` | 66 | 24 | 11.2% | 100% |

With `splitter="safe"` the two are close, because atom-level tokens can't absorb a ring
pair either way. The separation opens up with `splitter=none`, where BPE merges can
swallow both halves of a pair into one token — and that is where recap pulls ahead
(11.2% vs 3.2% of pairs made unbreakable).

The other four slicers were ruled out on validity, not span: `hr` 48%, `rotatable` 64%,
`mmpa` 66% of their SAFE strings fail to decode back to a molecule at all.